# Deepfake Video Detection Using Deep Learning.

In [3]:
# ---
# 1. IMPORTS & CONFIGURATION
# Set up libraries and basic parameters.
# ---
# Requirements:
# %pip install torch torchvision matplotlib
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch # Core Deep Learning library
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights # Pre-trained weights

import warnings
warnings.filterwarnings('ignore')

# Configuration Variables
DATA_DIR = os.path.join("data", "Faces")
CELEBDF_DIR = os.path.join("data", "CelebDF_Faces")
BATCH_SIZE = 4
# Train for 2 epochs only
EPOCHS = 2
NUM_FRAMES = 16 # exactly 16 frames per video to capture temporal context

# Automatically use GPU if available, otherwise CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cpu


In [4]:
# Quick sample check
def show_sample_faces(data_dir, n=4):
    # Check both classes
    for label in ['Real', 'Fake']:
        label_dir = os.path.join(data_dir, label)
        print(f'\n{label}:')
        if not os.path.isdir(label_dir):
            print('Missing folder')
            continue
        # Check a few folders
        for folder in sorted(os.listdir(label_dir))[:n]:
            path = os.path.join(label_dir, folder, 'frame_00.jpg')
            print(folder, 'OK' if os.path.isfile(path) else 'Missing')

show_sample_faces(DATA_DIR)



Real:
008 OK
033 OK
035 OK
036 OK

Fake:
008_990 OK
033_097 OK
035_036 OK
036_035 OK


In [5]:
# ---
# 3. DATASET & SPLITTING
# DeepFakeDataset loads 16 spatial frames per video.
# ---
IMAGE_SIZE = 224
LABEL_MAP = {"Real": 0, "Fake": 1}

class DeepFakeDataset(Dataset):
    def __init__(self, video_paths, labels, mode="train"):
        self.video_paths = video_paths
        self.labels = labels
        self.mode = mode
        
        # Color augmentation helps prevent overfitting during training
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ColorJitter(brightness=0.1, contrast=0.1) if mode == "train" else transforms.Lambda(lambda x: x),
            transforms.GaussianBlur(kernel_size=3) if mode == "train" else transforms.Lambda(lambda x: x),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        sample_dir = self.video_paths[idx]
        label = self.labels[idx]
        do_flip = self.mode == "train" and random.random() > 0.5
        
        tensors = []
        # Loop to load exactly 16 frames per video sequence
        for i in range(NUM_FRAMES):
            frame_path = os.path.join(sample_dir, f"frame_{i:02d}.jpg")
            if not os.path.isfile(frame_path):
                return None, None  # Skip incomplete sequences
            
            img = Image.open(frame_path).convert("RGB")
            if do_flip: img = transforms.functional.hflip(img)
            tensors.append(self.transform(img))
            
        # Stack 16 images into a single tensor block
        return torch.stack(tensors), torch.tensor(label, dtype=torch.float32)

def safe_collate(batch):
    # Safely filters out (None, None) invalid samples to prevent crashes
    batch = [b for b in batch if b[0] is not None]
    if not batch: return None, None
    tensors, labels = zip(*batch)
    return torch.stack(tensors), torch.stack(labels)

# Scan directories and prepare data paths
video_paths, labels = [], []
for lbl in ["Real", "Fake"]:
    lbl_dir = os.path.join(DATA_DIR, lbl)
    if os.path.isdir(lbl_dir):
        for f in os.listdir(lbl_dir):
            if len(os.listdir(os.path.join(lbl_dir, f))) >= NUM_FRAMES:
                video_paths.append(os.path.join(lbl_dir, f))
                labels.append(LABEL_MAP[lbl])

# Split: 80% for training, 20% for validation
split_idx = int(len(video_paths) * 0.8)
train_loader = DataLoader(DeepFakeDataset(video_paths[:split_idx], labels[:split_idx], "train"), batch_size=BATCH_SIZE, collate_fn=safe_collate, shuffle=True)
val_loader = DataLoader(DeepFakeDataset(video_paths[split_idx:], labels[split_idx:], "val"), batch_size=BATCH_SIZE, collate_fn=safe_collate, shuffle=False)

print(f"Train samples: {split_idx}, Val samples: {len(video_paths) - split_idx}")


Train samples: 240, Val samples: 60


In [ ]:
# ---
# 4. MODEL ARCHITECTURE
# Transfer Learning using ResNet18.
# Extracts spatial features from frames and aggregates via Mean Pooling.
# ---
class DeepFakeDetector(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        # Load pre-trained ResNet18 to leverage existing visual knowledge
        self.backbone = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        
        # Freeze early layers to retain generic features (edges, shapes)
        for name, param in self.backbone.named_parameters():
            if "layer4" not in name:
                param.requires_grad = False
                
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        # Custom Classification Head for Binary output (Real vs Fake)
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout), # Dropout prevents over-reliance on specific neurons
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

model = DeepFakeDetector().to(DEVICE)
print("Model initialized and moved to device.")


Model initialized and moved to device.


In [ ]:
# ---
# 5. TRAINING LOOP + VALIDATION
# Final training setting: 2 epochs
# ---
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=5e-4
)

print(f"Starting training for {EPOCHS} epochs...")

# Train the model for the number of epochs defined above
for epoch in range(1, EPOCHS + 1):
    # ---- TRAIN ----
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for inputs, targets in train_loader:
        if inputs is None:
            continue

        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE).float()

        b, f, c, h, w = inputs.shape
        inputs_flat = inputs.view(b * f, c, h, w)

        # Reset gradients before the next optimisation step
        optimizer.zero_grad()
        logits_flat = model(inputs_flat).squeeze(1)
        video_logits = logits_flat.view(b, f).mean(dim=1)

        loss = criterion(video_logits, targets)
        loss.backward()
        # Update trainable weights
        optimizer.step()

        train_loss += loss.item() * b
        preds = torch.sigmoid(video_logits) > 0.5
        train_correct += (preds == (targets > 0.5)).sum().item()
        train_total += b

    train_acc = train_correct / train_total if train_total > 0 else 0
    train_loss = train_loss / train_total if train_total > 0 else 0

    # ---- VALIDATION ----
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    tp, tn, fp, fn = 0, 0, 0, 0

    with torch.no_grad():
        for inputs, targets in val_loader:
            if inputs is None:
                continue

            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE).float()

            b, f, c, h, w = inputs.shape
            inputs_flat = inputs.view(b * f, c, h, w)

            logits_flat = model(inputs_flat).squeeze(1)
            video_logits = logits_flat.view(b, f).mean(dim=1)

            loss = criterion(video_logits, targets)
            preds = torch.sigmoid(video_logits) > 0.5
            truth = targets > 0.5

            val_loss += loss.item() * b
            val_correct += (preds == truth).sum().item()
            val_total += b

            tp += ((preds == 1) & (truth == 1)).sum().item()
            tn += ((preds == 0) & (truth == 0)).sum().item()
            fp += ((preds == 1) & (truth == 0)).sum().item()
            fn += ((preds == 0) & (truth == 1)).sum().item()

    val_acc = val_correct / val_total if val_total > 0 else 0
    val_loss = val_loss / val_total if val_total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # Show current progress so it is easy to follow the training stage
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")


In [ ]:
# ---
# 6. CROSS-DATASET EVALUATION (Celeb-DF)
# Evaluate the model on Celeb-DF.
# ---
celeb_paths, celeb_labels = [], []
for lbl in ["Real", "Fake"]:
    lbl_dir = os.path.join(CELEBDF_DIR, lbl)
    if os.path.isdir(lbl_dir):
        for f in os.listdir(lbl_dir):
            if len(os.listdir(os.path.join(lbl_dir, f))) >= NUM_FRAMES:
                celeb_paths.append(os.path.join(lbl_dir, f))
                celeb_labels.append(LABEL_MAP[lbl])


celeb_loader = DataLoader(DeepFakeDataset(celeb_paths, celeb_labels, "val"), batch_size=BATCH_SIZE, collate_fn=safe_collate)

model.eval() # Turn off dropout/batchnorm for evaluation
correct, total = 0, 0
fp, fn, tp, tn = 0, 0, 0, 0

with torch.no_grad(): # Disable gradient calculation to save memory
    for inputs, targets in celeb_loader:
        if inputs is None: continue
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE).float()
        
        b, f, c, h, w = inputs.shape
        inputs_flat = inputs.view(b * f, c, h, w)
        logits_flat = model(inputs_flat).squeeze(1)
        video_logits = logits_flat.view(b, f).mean(dim=1)
        
        preds = torch.sigmoid(video_logits) > 0.5
        truth = targets > 0.5
        
        correct += (preds == truth).sum().item()
        total += b
        
        # Calculate Confusion Matrix components
        tp += ((preds == 1) & (truth == 1)).sum().item() # True Positive (Fake detected as Fake)
        tn += ((preds == 0) & (truth == 0)).sum().item() # True Negative (Real detected as Real)
        fp += ((preds == 1) & (truth == 0)).sum().item() # False Positive
        fn += ((preds == 0) & (truth == 1)).sum().item() # False Negative

if total > 0:
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Celeb-DF Accuracy: {correct/total:.4f}")
    print(f"Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"Confusion Matrix: TP={tp}, TN={tn}, FP={fp}, FN={fn}")
else:
    print("No complete Celeb-DF evaluation data found in the directory.")
